In [1]:
import os
from tqdm.notebook import tqdm
import numpy as np
from clustpy.deep.neural_networks.feedforward_autoencoder import FeedforwardAutoencoder
from sklearn.cluster import KMeans
from SHiP import SHiP
from helper import (
    load_data,

    load_pendigits,
    load_optdigits,
    load_letterrecognition,
    load_gaussian_blobs,
    load_example,
    load_usps,
    load_htru,
    load_har,
    load_mice,
    load_synth_high,
    load_synth_low,
    load_mnist,
    load_fmnist,
    load_coil20,
    load_coil100,
    load_weizmann,

    detect_device,
    save_dict_as_json,
    load_json_as_dict,
    load_pretrained_model,
    find_local_core_points_same,

    encode_batchwise,
    get_train_and_testloader,
)

[WARNING] Could not import nltk in clustpy.data.real_world_data. Please install nltk by 'pip install nltk' if necessary


In [8]:
datasets_loading_methods = [
    load_pendigits,
    load_optdigits,
    load_letterrecognition,
    load_gaussian_blobs,
    load_example,
    load_usps,
    load_htru,
    load_har,
    load_mice,
    load_synth_high,
    load_synth_low,
    
    load_mnist,
    load_fmnist,
    load_coil20,
    load_coil100,
    load_weizmann,
]

In [13]:
device = detect_device()
experiments_path = "/export/share/peters57dm/Verbund/deepsync/experiments"
pretrained_models_path = "/export/share/peters57dm/Verbund/data/n-pretrained-models/mse_pretrained_models256-128-64"
experiment_name = "corepoints_effect"
batch_size = 256
max_embed_size = 10
ae_layers = [256, 128, 64]
k = 25
percent = 0.1
N_MODELS = 5

In [ ]:
results = {}
for ds_loader in tqdm(datasets_loading_methods, total=len(datasets_loading_methods)):
    try :
            data, gt_labels, data_name = load_data(ds_loader)
            trainloader, testloader = get_train_and_testloader(data, gt_labels, batch_size)
            print(f"Dataset: {data_name}")
    except : # handling any reason for this particular dataset loading failure
        print(f"Loading {data_name} failed. Continue to next experiment")
        continue
    embedded_space_dim = min(data.shape[1], max_embed_size)
    gt_labels = gt_labels.numpy()
    for i in range(N_MODELS):   
        # initialize results dictionary
        results[f"{data_name}_{i}"] =  {}
        
        # load the data and model
        _mpath = os.path.join(pretrained_models_path, f"pretrained_{data_name}_{i}.pth")
        
        k_true = len(np.unique(gt_labels))
        model = FeedforwardAutoencoder(layers=[data.shape[1],
                                                ae_layers[0], ae_layers[1], ae_layers[2],
                                                embedded_space_dim]).to(device)
        model = load_pretrained_model(model, _mpath, device)
        embedded, gt_labels = encode_batchwise(testloader, model, device)
        core_points_mask, th = find_local_core_points_same(embedded, k, percent)
        core_points = embedded[np.where(np.diag(core_points_mask)==1)[0]]
        k_true_core = len(np.unique(gt_labels[np.where(np.diag(core_points_mask)==1)[0]]))

        # ship core
        ship_core = SHiP(data=core_points, treeType="DCTree")
        ship_core_labels = ship_core.fit_predict(power=2, partitioningMethod="ThreshholdElbow")
        results[f"{data_name}_{i}"]["ship_core_labels"] = ship_core_labels

        # ship embedded
        ship_embedded = SHiP(data=embedded, treeType="DCTree")
        ship_embedded_labels = ship_embedded.fit_predict(power=2, partitioningMethod="ThreshholdElbow")
        results[f"{data_name}_{i}"]["ship_embedded_labels"] = ship_embedded_labels

        # kmeans core
        kmeans_core = KMeans(n_clusters=k_true_core, init="k-means++", n_init=10)
        kmeans_core_labels = kmeans_core.fit_predict(core_points)
        results[f"{data_name}_{i}"]["kmeans_core_labels"] = kmeans_core_labels

        # kmeans embedded
        kmeans_embedded = KMeans(n_clusters=k_true, init="k-means++", n_init=10)
        kmeans_embedded_labels = kmeans_embedded.fit_predict(embedded)
        results[f"{data_name}_{i}"]["kmeans_embedded_labels"] = kmeans_embedded_labels
save_dict_as_json(results, os.path.join(experiments_path, experiment_name, "kmeans_ship_labels.json"))

  0%|          | 0/16 [00:00<?, ?it/s]

Dataset: pendigits


/export/share/peters57dm/Verbund/deepsync/helper.py:775: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_model_path, map_location=device)
/e

Dataset: optdigits


/export/share/peters57dm/Verbund/deepsync/helper.py:775: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_model_path, map_location=device)
/e

...

/export/share/peters57dm/Verbund/deepsync/helper.py:775: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_model_path, map_location=device)
/e

Dataset: MNIST


/export/share/peters57dm/Verbund/deepsync/helper.py:775: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_model_path, map_location=device)
/e

Dataset: FMNIST


/export/share/peters57dm/Verbund/deepsync/helper.py:775: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_model_path, map_location=device)
/e

# Results

In [28]:
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score

# Assuming your data is loaded into `results_dict`
# Example: results_dict = {'pendigits_0': {'ship_core_labels': [...], ...}, ...}

def compute_metrics(results_dict):
    # Group keys by dataset
    dataset_groups = {}
    for key in results_dict:
        dataset_name = "_".join(key.split("_")[:-1])
        dataset_groups.setdefault(dataset_name, []).append(key)

    # Initialize result storage
    rows = []

    for dataset, keys in sorted(dataset_groups.items()):
        ship_core_ami, ship_core_ari = [], []
        kmeans_core_ami, kmeans_core_ari = [], []
        ship_emb_ami, ship_emb_ari = [], []
        kmeans_emb_ami, kmeans_emb_ari = [], []

        for key in keys:
            data = results_dict[key]

            # Compute scores
            ship_core_ami.append(adjusted_mutual_info_score(data['ship_core_labels'], data['true_core_labels']))
            ship_core_ari.append(adjusted_rand_score(data['ship_core_labels'], data['true_core_labels']))

            kmeans_core_ami.append(adjusted_mutual_info_score(data['kmeans_core_labels'], data['true_core_labels']))
            kmeans_core_ari.append(adjusted_rand_score(data['kmeans_core_labels'], data['true_core_labels']))

            ship_emb_ami.append(adjusted_mutual_info_score(data['ship_embedded_labels'], data['true_embedded_labels']))
            ship_emb_ari.append(adjusted_rand_score(data['ship_embedded_labels'], data['true_embedded_labels']))

            kmeans_emb_ami.append(adjusted_mutual_info_score(data['kmeans_embedded_labels'], data['true_embedded_labels']))
            kmeans_emb_ari.append(adjusted_rand_score(data['kmeans_embedded_labels'], data['true_embedded_labels']))

        # Helper to format mean±std
        def fmt(x):
            return f"{np.mean(x):.3f}±{np.std(x):.3f}"

        # Append results row
        rows.append({
            "Dataset": dataset,
            "Ship Core AMI": fmt(ship_core_ami),
            "Ship Core ARI": fmt(ship_core_ari),
            "KMeans Core AMI": fmt(kmeans_core_ami),
            "KMeans Core ARI": fmt(kmeans_core_ari),
            "Ship Emb AMI": fmt(ship_emb_ami),
            "Ship Emb ARI": fmt(ship_emb_ari),
            "KMeans Emb AMI": fmt(kmeans_emb_ami),
            "KMeans Emb ARI": fmt(kmeans_emb_ari),
        })

    # Create DataFrame
    df = pd.DataFrame(rows)
    df = df.sort_values(by="Dataset").reset_index(drop=True)

    # Reorder columns
    column_order = [
        "Dataset",
        "Ship Core AMI", "Ship Emb AMI",
        "Ship Core ARI", "Ship Emb ARI",
        "KMeans Core AMI", "KMeans Emb AMI",
        "KMeans Core ARI", "KMeans Emb ARI",
    ]
    df = df[column_order]
    return df

# Example usage:
# df_results = compute_metrics(results_dict)
# print(df_results.to_string(index=False))


In [29]:
results_path = "/export/share/peters57dm/Verbund/deepsync/experiments/corepoints_effect/correct_kmeans_ship_labels.json"
experiment_labels = load_json_as_dict(results_path)

In [30]:
len(experiment_labels['pendigits_0']['kmeans_embedded_labels'])

10992

In [31]:
df_results = compute_metrics(experiment_labels)
print(df_results.to_string(index=False))

          Dataset Ship Core AMI Ship Emb AMI Ship Core ARI Ship Emb ARI KMeans Core AMI KMeans Emb AMI KMeans Core ARI KMeans Emb ARI
           FMNIST   0.707±0.017  0.529±0.028   0.540±0.032  0.324±0.024     0.673±0.021    0.579±0.013     0.531±0.019    0.403±0.016
              HAR   0.757±0.032  0.608±0.022   0.611±0.063  0.471±0.029     0.769±0.023    0.668±0.022     0.680±0.049    0.566±0.029
            MNIST   0.922±0.012  0.720±0.025   0.885±0.045  0.635±0.042     0.873±0.013    0.723±0.005     0.751±0.020    0.649±0.018
             USPS   0.937±0.022  0.510±0.007   0.933±0.027  0.330±0.068     0.893±0.012    0.658±0.012     0.795±0.027    0.564±0.014
          coil100   0.968±0.002  0.880±0.009   0.912±0.021  0.669±0.084     0.936±0.002    0.800±0.005     0.855±0.008    0.559±0.013
           coil20   0.945±0.009  0.866±0.006   0.896±0.019  0.755±0.013     0.918±0.017    0.756±0.009     0.863±0.028    0.580±0.022
       easy_blobs   1.000±0.000  0.999±0.001   1.000±0.000  1.

In [33]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils.dataframe import dataframe_to_rows

def save_df_to_styled_excel(df, filename="results_table.xlsx"):
    wb = Workbook()
    ws = wb.active
    ws.title = "Experiment Results"

    # Styles
    header_font = Font(bold=True)
    fill_odd = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
    border = Border(
        left=Side(border_style="thin", color="000000"),
        right=Side(border_style="thin", color="000000"),
        top=Side(border_style="thin", color="000000"),
        bottom=Side(border_style="thin", color="000000")
    )

    # Write DataFrame to worksheet
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), start=1):
        for c_idx, value in enumerate(row, start=1):
            cell = ws.cell(row=r_idx, column=c_idx, value=value)

            # Apply styles
            cell.border = border
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

            if r_idx == 1:  # Header
                cell.font = header_font
            elif r_idx % 2 == 0:  # Even data rows (zebra stripe)
                cell.fill = fill_odd

    # Adjust column widths
    for col in ws.columns:
        max_length = 0
        col_letter = col[0].column_letter
        for cell in col:
            try:
                max_length = max(max_length, len(str(cell.value)))
            except:
                pass
        ws.column_dimensions[col_letter].width = max_length + 2

    # Save the workbook
    wb.save(filename)
    print(f"Saved styled Excel file as: {filename}")


In [ ]:
save_df_to_styled_excel(df_results, filename=os.path.join(experiments_path, experiment_name, f"{experiment_name}_results_table.xlsx"))

Saved styled Excel file as: /export/share/peters57dm/Verbund/deepsync/experiments/corepoints_effect/results_table.xlsx


In [31]:
from helper import detect_device, load_json_as_dict, load_letterrecognition, load_data, encode_batchwise, get_train_and_testloader, load_pretrained_model
import torch
from clustpy.deep.neural_networks.feedforward_autoencoder import FeedforwardAutoencoder
import numpy as np
import os

In [32]:
res = load_json_as_dict("/export/share/peters57dm/Verbund/deepsync/experiments/comparison402/ae_sync_loss/knn_label_assignment/letterrecognition/exp_00/trackers/eval_tracker.json")
device = detect_device()
pretrained_model_path = "/export/share/peters57dm/Verbund/data/n-pretrained-models/mse_pretrained_models256-128-64"

In [34]:
data, gt_labels, data_name = load_data(load_letterrecognition)
model = FeedforwardAutoencoder(layers=[data.shape[1],
                                        256, 128, 64,
                                        10]).to(device)
_mpath = os.path.join(pretrained_model_path, f"pretrained_{data_name}_{0}.pth")
model = load_pretrained_model(model, _mpath, device)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-4)
trainloader, testloader = get_train_and_testloader(data, gt_labels, 256)
embedded, gt_labels = encode_batchwise(testloader, model, device)
true_k = len(torch.unique(gt_labels))

In [39]:
from SHiP import SHiP
ship = SHiP(data=embedded, treeType="DCTree", config={"k":true_k})
ship_labels = ship.fit_predict(power=2, partitioningMethod="K")

In [40]:
np.unique(ship_labels)

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25])